In [ ]:
from __future__ import annotations
import os
import csv
from typing import List, Dict, Iterable, Literal, Optional



In [ ]:
import os
import requests
API_KEY = "__YOUR_API_KEY__"
MODEL = "gemini-2.0-flash"
URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def gemini_prompt(prompt: str) -> str:
    """Send a prompt to Gemini API and return the generated text."""
    if not API_KEY:
        raise ValueError("Missing GEMINI_API_KEY. Please set it as an environment variable.")

    headers = {
        "Content-Type": "application/json",
        "X-goog-api-key": API_KEY,
    }

    payload = {
        "contents": [
            {
                "parts": [{"text": prompt}]
            }
        ]
    }

    resp = requests.post(URL, headers=headers, json=payload, timeout=30)
    resp.raise_for_status()

    data = resp.json()
    try:
        return data["candidates"][0]["content"]["parts"][0]["text"]
    except Exception:
        return str(data)  # fallback: dump raw response if unexpected

In [ ]:
import pandas as pd
import time
import random
import requests  # For handling requests exceptions (like rate limit errors)

# Function to handle retries with exponential backoff
def exponential_backoff(request_func, max_retries=5, base_delay=30):
    retries = 0
    while retries < max_retries:
        try:
            # Call the request function
            return request_func()
        except requests.exceptions.RequestException as e:  # Catching request-related exceptions
            print(f"Request failed: {e}. Retrying in {base_delay * (2 ** retries)} seconds...")
            time.sleep(base_delay * (2 ** retries))  # Exponential backoff
            retries += 1
            if retries == max_retries:
                print("Max retries reached. Exiting.")
                raise e
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            break

# Your API call wrapped in a function
def get_translation(text,test):
    prompt = f"""Translate the following Bangla Python Code Instruction to English and only return the English translation. Do not change the example function and parameter names and only update the function parameter types and return variable types of Example function prototype to actual python syntax based on the provided unit test. Do not give the full code implementation. Just give the updated prototype.

Use the following glossary for translation:

**Arithmetic & Number Theory:**
- গ.সা.গু / গরিষ্ঠ সাধারণ গুণনীয়ক → GCD
- ল.সা.গু / লঘিষ্ঠ সাধারণ গুণিতক → LCM
- যোগফল → sum
- বিয়োগফল → difference
- গুণফল → product
- ভাগফল → quotient
- মডুলাস / মডুলো → modulus/modulo
- শক্তি / ঘাত → power
- মূল → root
- বর্গমূল → square root
- ঘনমূল → cube root
- অবশিষ্ট → remainder
- মৌলিক সংখ্যা / প্রাইম নাম্বার → prime number
- যৌগিক সংখ্যা / কম্পোজিট নাম্বার → composite number
- সমান → equal
- সমান নয় → not equal
- ছোট → less than
- বড় → greater than
- সমান বা ছোট → less than or equal to
- সমান বা বড় → greater than or equal to
- বিজোড় → odd
- যুগ্ম → even
- গুণনীয়ক → factor
- গুণিতক → multiple
- ভাজক / ডিভাইসর → divisor
- গুণ → multiply
- ভাগ → divide
- যোগ → add
- বিয়োগ → subtract

**Programming Flow Control:**
- যদি / ইফ → if
- অন্যথা / এলস → else
- যদি না / এলস ইফ → else if
- লুপ / লুপ করো → loop
- যতক্ষণ / হোয়াইল → while
- জন্য / ফর → for
- থামাও / ব্রেক → break
- অগ্রসর হও / কন্টিনিউ → continue
- ফেরত / রিটার্ন → return

**Data Structures:**
- তালিকা / লিস্ট → list
- অ্যারে → array
- অ্যারে লিস্ট → array list
- স্ট্যাক / পাইল → stack
- কিউ / পংক্তি → queue
- ডেক / ডিক → deque
- সেট / সমষ্টি → set
- ম্যাপ → map
- ডিকশনারি / অভিধান → dictionary
- হ্যাশম্যাপ → hashmap
- গ্রাফ / ছক → graph
- ট্রি / গাছ → tree
- বাইনারি ট্রি / বাইনারি গাছ → binary tree
- বাইনারি সার্চ ট্রি / বাইনারি অনুসন্ধান গাছ → binary search tree
- হীপ → heap
- প্রায়োরিটি কিউ / অগ্রাধিকার কিউ → priority queue
- ডিএফএস / গভীরতা প্রথম অনুসন্ধান → DFS
- বিএফএস / প্রস্থ প্রথম অনুসন্ধান → BFS

**Sorting & Searching:**
- সাজাও / বাছাই → sort
- ক্রমবর্ধমান → ascending
- ক্রমহ্রাসমান → descending
- অনুসন্ধান / সার্চ → search
- বাইনারি সার্চ / বাইনারি অনুসন্ধান → binary search
- লিনিয়ার সার্চ / রৈখিক অনুসন্ধান → linear search

**String Handling:**
- স্ট্রিং / পাঠ্য → string
- উল্টো / বিপরীত → reverse
- সাবস্ট্রিং / উপস্ট্রিং → substring
- সংযুক্ত / সংযোজন → concatenate
- দৈর্ঘ্য / আয়তন → length
- অক্ষর / চরিত্র → character
- বড় হাতের → uppercase
- ছোট হাতের → lowercase
- প্যালিনড্রোম / সমপাঠ্য → palindrome

**Input/Output & Miscellaneous:**
- ইনপুট / প্রবেশ → input
- আউটপুট / বাহির → output
- প্রিন্ট → print
- পড়ো / পাঠ → read
- লিখো → write
- ফাইল / নথি → file
- সময় জটিলতা → time complexity
- স্থান জটিলতা → space complexity
- অ্যালগরিদম / পদ্ধতি → algorithm
- টেস্টকেস / পরীক্ষা / পরীক্ষা কেস → test case
- ইনডেক্স / সূচক → index
- ইটারেট / পুনরাবৃত্তি করো → iterate
- পুনরাবৃত্তি / রিকারশন → recursion

Bangla: {text}
Unit Test: {test}
"""
    llm_response = gemini_prompt(prompt=prompt)
    return llm_response


In [ ]:
import time
import pandas as pd
import os

df = pd.read_csv("test_v1.csv")  # Try the simple read first
# print(df["instruction"][46])
# print(df["test_list"][46])
# print(get_translation(df["instruction"][46],df["test_list"][0]))

#Prepare texts (handle NaN & ensure strings)


# Assuming you already have the DataFrame df loaded
texts = df["instruction"].fillna("").astype(str).tolist()
unit_tests = df["test_list"].fillna("").astype(str).tolist()

translated = []
checkpoint_file = "translation_checkpoint.csv"

# Check if checkpoint file exists, and if so, load the translated entries
if os.path.exists(checkpoint_file):
    checkpoint_df = pd.read_csv(checkpoint_file)
    translated = checkpoint_df["instruction_en"].tolist()
    start_index = len(translated)
else:
    start_index = 0

# Loop through the remaining texts
for i in range(start_index, len(texts)):
    try:
        out = exponential_backoff(lambda: get_translation(texts[i], unit_tests[i]))  # Use exponential backoff
        print(texts[i])
        print(out)
        translated.append(out)  # Append translated text
        
        # Periodically save the translations to a checkpoint file
        if (i + 1) % 50 == 0:  # Checkpoint every 50 translations
            checkpoint_df = pd.DataFrame({"instruction_en": translated})
            checkpoint_df.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved at index {i + 1}")

        time.sleep(10)  # Small sleep to avoid immediate re-request
    except Exception as e:
        print(f"Failed to get response for index {i}: {e}")

# After the loop, save the final results to the dataframe and write out the full CSV
df["instruction_en"] = translated
df.to_csv("test_v1_en_gemini.csv", index=False)

# Save the final checkpoint as well
checkpoint_df = pd.DataFrame({"instruction_en": translated})
checkpoint_df.to_csv(checkpoint_file, index=False)
print("Final checkpoint saved.")
